In [1]:
import math
import numpy as np

def prom_update(prom_prev: float, x_new: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return x_new
    else:
        return prom_prev + (x_new - prom_prev) / n_curr

def s_cuad_update(s_cuad_prev: float, prom_prev: float, prom_curr: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return 0.0
    else:
        return (1 - (1/(n_curr - 1))) * s_cuad_prev + n_curr * ((prom_curr - prom_prev)**2)


### Ejercicio 7.
Considerar un sistema de colas con un único servidor que recibe solicitudes de acuerdo con un proceso de Poisson no homogéneo, cuya tasa es inicialmente de $4$ solicitudes por hora, aumenta linealmente hasta alcanzar $19$ solicitudes por hora luego de $5$ horas, y posteriormente disminuye linealmente hasta volver a $4$ solicitudes por hora luego de otras $5$ horas. Este comportamiento de la tasa se repite indefinidamente; es decir,

$$ \lambda(t + 10) = \lambda(t) $$

- Suponer además que el tiempo de servicio del servidor se distribuye de manera exponencial, con una tasa de $13$ servicios por hora .

Se pide:

a. Desarrollar un programa que simule el proceso durante un tiempo T = 100 horas, registrando los tiempos de llegada, los tiempos de servicio y la evolución del número de trabajos en cola.

b. Utilizar el programa para realizar simulaciones y estimar el tiempo promedio que tarda en ser procesada una solicitud, desde que llega al sistema hasta que finaliza el proceso. Detener las simulaciones cuando la desviación estándar del estimador de la media muestral sea inferior a $0.01$

c. Utilizar el programa para realizar simulaciones y estimar la probabilidad de que haya solicitudes que se
completen luego del tiempo T . Detener las simulaciones cuando la desviación estándar del estimador de
la proporción sea inferior a $0.01$

In [ ]:
# Punto A

def lamda_t(t: float):
    t_m = t % 10

    if t_m <= 5:
        return 3*t_m + 4
    else:
        return -3*t_m + 34

def t_nueva_solicitud():
    lamda_cota = 19
    t = 0

    while True:
        t -= math.log(1 - np.random.random()) / lamda_cota
        V = np.random.random()

        if V < lamda_t(t) / lamda_cota:
            return t

def t_servicio():
    # Exponencial con lambda = 13 (13 servicios por hora)
    return -math.log(1 - np.random.random()) / 13

def punto_a(T: int = 100):
    # Registros
    A = [] # Tiempo en la que llega una solicitud
    D = [] # Tiempo en el cual se procesa una solicitud
    N = [0] # Cola de solicitudes
    N_T = [0] # Tiempo en el cual se cambia la cola de solicitudes

    # Inicialización
    t, NA, ND = 0, 0, 0
    n = 0

    tA = t + t_nueva_solicitud()
    tD = math.inf

    while tA < math.inf or tD < math.inf:
        evento_prox = min(tA, tD)

        # Llega nuevo solicitud
        if evento_prox == tA:
            t = tA
            NA += 1
            n += 1

            A.append(t)
            N.append(n)
            N_T.append(t)

            tA = t + t_nueva_solicitud()
            if tA > T:
                # El tiempo de la nueva solicitud supero T, por lo que no se
                # reciben nuevas solicitudes y solo se procesa las solicitudes
                # en la cola
                tA = math.inf

                # print(
                #    f"Ya pasaron las {T} horas, por lo que ya no llegan mas",
                #    f"solicitudes. Quedan {n} clientes por atender.\n")

            if n == 1:
                tD = t + t_servicio()

        elif evento_prox == tD:
            # Se termina de procesar una solicitud
            t = tD
            ND += 1
            n -= 1

            D.append(t)
            N.append(n)
            N_T.append(t)

            if n > 0:
                # Se añade el siguiente evento de atención del servidor
                tD = t + t_servicio()
            else:
                # Se vacio la cola, por lo que, el servidor entra en reposo
                tD = math.inf

    return t, A, D, N, N_T

# t, A, D, N, _ = punto_a()
# print(f"Simulación finalizada a las {t:.2f} horas")
# print(f"Total de llegadas registradas: {len(A)}")
# print(f"Total de servicios completados: {len(D)}")
# print(f"Total de descansos tomados: {len(R)}")
# print()

# print("Tiempos de llegada (A):", A[:10], "...")
# print("Tiempos de salida (D):", D[:10], "...")
# print("Evolución de la cola (N):", N[:10], "...")

In [3]:
# Punto B
def generar_media_i():
    _, A_i, D_i, _, _ = punto_a()

    return (np.array(D_i) - np.array(A_i)).mean()

def punto_b():
    media = generar_media_i()
    s_cuad, n = 0, 1

    while n < 100 or math.sqrt(s_cuad / n) >= 0.01:
        X = generar_media_i()
        n += 1

        media_prev = media
        media = prom_update(media, X, n)
        s_cuad = s_cuad_update(s_cuad, media_prev, media, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Tiempo promedio que tarda una solicitud en ser procesada ~ {media:.4f} horas")
    print(f"Desviación estándar muestral = {math.sqrt(s_cuad / n):.4f} ")

punto_b()

Cantidad de datos generados:  100
Tiempo promedio que tarda una solicitud en ser procesada ~ 0.1149 horas
Desviación estándar muestral = 0.0010 


In [ ]:
# Punto C
def generar_p_i():
    _, _, D_i, _, _ = punto_a()

    return (np.array(D_i) > 100.0).mean()

def punto_c():
    p = generar_p_i()
    n = 1

    while n < 100 or math.sqrt(p * (1 - p) / n) >= 0.01:
        X = generar_p_i()
        n += 1

        p = prom_update(p, X, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Probabilidad de que las solicitudes se completen luego del timepo T es aprox {p:.4f}")
    print(f"Desviación estándar muestral = {math.sqrt(p * (1 - p) / n):.4f} ")

punto_c()

Cantidad de datos generados:  100
Probabilidad de que las solicitudes se completen luego del timepo T es ~ 0.0014 horas
Desviación estándar muestral = 0.0037 
